# Phase 3: Alpha Research & Statistical Learning  
## Notebook 03_08 — LOB Imbalance and Microprice

### Research question

Can top-of-book and limit-order-book imbalance features improve short-horizon mid-price direction forecasts, and how should microprice be computed without overclaiming predictive power?

This notebook follows naturally from:

```text
01_09_tick_to_bar_sampling_bias
03_04_gmm_hmm_tick_regime_detection
```

The earlier tick notebook was deliberately constrained to **L1 bid/ask** data. This notebook expands the microstructure toolkit.

It handles two data cases:

### Case A — L1 top-of-book only

```text
timestamp, bid1, ask1, bid1_size, ask1_size
```

Available features:

- mid price;
- spread;
- top-of-book imbalance;
- microprice;
- microprice displacement;
- short-horizon labels.

### Case B — multi-level LOB snapshot

```text
bid1..bidN, ask1..askN, bid1_size..bidN_size, ask1_size..askN_size
```

Available additional features:

- depth-weighted imbalance;
- level-by-level imbalance;
- weighted microprice;
- book pressure;
- near/far book liquidity features.

The key idea:

> Microprice and imbalance are not magic alpha. They are short-horizon microstructure features that must be tested with strict timestamp alignment, realistic costs, and no look-ahead.

## 1. Top-of-book mid, spread, and imbalance

Best bid and ask:

$$
B_t = bid1_t
$$

$$
A_t = ask1_t
$$

Mid price:

$$
M_t = \frac{A_t+B_t}{2}
$$

Spread:

$$
S_t = A_t-B_t
$$

Top-of-book imbalance:

$$
I_t^{(1)} = \frac{Q^b_{1,t}-Q^a_{1,t}} {Q^b_{1,t}+Q^a_{1,t}}
$$

where:

- $Q^b_{1,t}$ is bid1 size;
- $Q^a_{1,t}$ is ask1 size.

Interpretation:

- positive imbalance: more visible size at best bid than best ask;
- negative imbalance: more visible size at best ask than best bid.

But this is **visible top-of-book imbalance only**.

## 2. Microprice

The microprice shifts the mid toward the side with less displayed liquidity.

A common top-of-book microprice is:

$$
\mu_t = \frac{ A_t Q^b_{1,t} + B_t Q^a_{1,t} } { Q^b_{1,t} + Q^a_{1,t} }
$$

If bid size is large relative to ask size, the microprice moves closer to the ask.

Microprice displacement:

$$
d_t = \mu_t - M_t
$$

Normalised displacement:

$$
\tilde d_t = \frac{\mu_t-M_t}{S_t}
$$

This feature is often interpreted as a queue-pressure proxy.

Important caution:

> Microprice is not a guarantee of the next tick. It is a short-horizon conditional feature that must be evaluated statistically.

## 3. Multi-level LOB imbalance

If we have $N$ levels of book depth, a simple depth imbalance is:

$$
\begin{aligned}
I_t^{(N)} &= \frac{ \sum_{\ell=1}^{N} Q^b_{\ell,t} - \sum_{\ell=1}^{N} Q^a_{\ell,t} } { \sum_{\ell=1}^{N} Q^b_{\ell,t} + \sum_{\ell=1}^{N} Q^a_{\ell,t} }
\end{aligned}
$$

A weighted version gives more weight to near-touch liquidity:

$$
\begin{aligned}
I_{w,t}^{(N)} &= \frac{ \sum_{\ell=1}^{N} w_\ell Q^b_{\ell,t} - \sum_{\ell=1}^{N} w_\ell Q^a_{\ell,t} } { \sum_{\ell=1}^{N} w_\ell Q^b_{\ell,t} + \sum_{\ell=1}^{N} w_\ell Q^a_{\ell,t} }
\end{aligned}
$$

where:

$$
w_\ell = \frac{1}{\ell}
$$

or another decreasing weight schedule.

This notebook simulates L5 snapshots so the full LOB features can be demonstrated, while still keeping an L1-only feature path.

## 4. Prediction target

We test short-horizon mid-price direction.

For horizon $h$:

$$
y_t = \operatorname{sign}(M_{t+h}-M_t)
$$

We also define a no-move class when the mid does not change.

For binary evaluation, we can focus on move events:

$$
y_t^{move}
=
\begin{cases}
1, & M_{t+h}>M_t \\
0, & M_{t+h}<M_t
\end{cases}
$$

This avoids pretending that no-move ticks are directional predictions.

## 5. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import roc_auc_score, accuracy_score, brier_score_loss
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

SKLEARN_AVAILABLE

In [ ]:
@dataclass(frozen=True)
class LOBConfig:
    n_ticks: int = 40_000
    n_levels: int = 5
    tick_size: float = 1.0
    base_price: float = 3000.0
    train_fraction: float = 0.70
    prediction_horizon_ticks: int = 10
    rolling_window: int = 200
    seed: int = 42


config = LOBConfig()
config

## 6. Synthetic L5 order book generator

We simulate a simplified L5 limit order book.

The synthetic process includes:

1. mid-price random walk with weak imbalance pressure;
2. variable spread;
3. bid/ask depths across five levels;
4. latent pressure variable that affects both imbalance and future price drift;
5. irregular timestamps.

This gives a controlled environment for testing feature construction and evaluation.

This is not a full exchange simulator. It does not model:

- order queue priority;
- hidden liquidity;
- individual order arrivals;
- cancellations;
- market-order flow;
- matching engine mechanics.

In [ ]:
def simulate_lob_snapshots(config: LOBConfig) -> pd.DataFrame:
    rng = np.random.default_rng(config.seed)

    n = config.n_ticks
    L = config.n_levels

    pressure = np.zeros(n)
    log_mid = np.zeros(n)
    spread_ticks = np.ones(n, dtype=int)
    intervals = np.zeros(n)

    log_mid[0] = np.log(config.base_price)

    bid_sizes = np.zeros((n, L))
    ask_sizes = np.zeros((n, L))

    for t in range(1, n):
        pressure[t] = 0.985 * pressure[t - 1] + 0.15 * rng.standard_normal()
        pressure[t] = np.clip(pressure[t], -3.0, 3.0)

        # Spread widens under high absolute pressure/noise.
        stress = abs(pressure[t]) + 0.3 * rng.standard_normal()
        if stress > 2.2:
            spread_ticks[t] = rng.choice([2, 3, 4], p=[0.55, 0.35, 0.10])
        elif stress > 1.2:
            spread_ticks[t] = rng.choice([1, 2, 3], p=[0.60, 0.35, 0.05])
        else:
            spread_ticks[t] = rng.choice([1, 2], p=[0.92, 0.08])

        # Weak short-horizon pressure drift plus noise.
        micro_drift = 0.000004 * pressure[t - 1]
        noise = 0.00008 * rng.standard_normal()
        jump = rng.normal(0, 0.0006) if rng.uniform() < 0.0008 else 0.0

        log_mid[t] = log_mid[t - 1] + micro_drift + noise + jump

        # More intense updates during high pressure.
        intervals[t] = rng.exponential(scale=max(0.05, 1.5 / (1 + abs(pressure[t]))))

        for level in range(L):
            decay = np.exp(-0.35 * level)
            base_depth = 60 * decay

            # Positive pressure increases bid depth and reduces ask depth near top.
            bid_mean = base_depth * np.exp(0.25 * pressure[t] * decay)
            ask_mean = base_depth * np.exp(-0.25 * pressure[t] * decay)

            bid_sizes[t, level] = rng.lognormal(np.log(max(bid_mean, 1.0)), 0.45)
            ask_sizes[t, level] = rng.lognormal(np.log(max(ask_mean, 1.0)), 0.45)

    intervals[0] = intervals[1]
    spread_ticks[0] = spread_ticks[1]
    pressure[0] = pressure[1]
    bid_sizes[0] = bid_sizes[1]
    ask_sizes[0] = ask_sizes[1]

    mid_raw = np.exp(log_mid)
    mid = np.round(mid_raw / config.tick_size) * config.tick_size

    spread = spread_ticks * config.tick_size
    bid1 = mid - spread / 2
    ask1 = mid + spread / 2

    # Ensure prices lie on a tick grid.
    bid1 = np.floor(bid1 / config.tick_size) * config.tick_size
    ask1 = bid1 + spread

    timestamps = pd.Timestamp("2024-01-02 09:00:00") + pd.to_timedelta(np.cumsum(intervals), unit="s")

    df = pd.DataFrame({
        "timestamp": timestamps,
        "latent_pressure": pressure,
        "mid_true": mid_raw,
        "bid1": bid1,
        "ask1": ask1,
        "spread_ticks_true": spread_ticks,
        "quote_interval_seconds": intervals
    })

    for level in range(1, L + 1):
        df[f"bid{level}"] = bid1 - (level - 1) * config.tick_size
        df[f"ask{level}"] = ask1 + (level - 1) * config.tick_size
        df[f"bid{level}_size"] = np.round(bid_sizes[:, level - 1]).clip(1)
        df[f"ask{level}_size"] = np.round(ask_sizes[:, level - 1]).clip(1)

    return df


lob_raw = simulate_lob_snapshots(config)

lob_raw.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(lob_raw["timestamp"], (lob_raw["bid1"] + lob_raw["ask1"]) / 2)
plt.title("Synthetic LOB Mid Price")
plt.xlabel("Timestamp")
plt.ylabel("Mid")
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(lob_raw["timestamp"], lob_raw["latent_pressure"])
plt.title("Latent Pressure Used Only for Synthetic Validation")
plt.xlabel("Timestamp")
plt.ylabel("Latent pressure")
plt.show()

## 7. L1 feature construction

These features require only:

```text
bid1, ask1, bid1_size, ask1_size
```

They are therefore compatible with a top-of-book dataset.

In [ ]:
def add_l1_microprice_features(df: pd.DataFrame, config: LOBConfig) -> pd.DataFrame:
    out = df.sort_values("timestamp").copy().reset_index(drop=True)

    out["mid"] = (out["bid1"] + out["ask1"]) / 2.0
    out["spread"] = out["ask1"] - out["bid1"]
    out["spread_ticks"] = out["spread"] / config.tick_size
    out["relative_spread"] = out["spread"] / out["mid"].clip(lower=1e-12)

    denom = out["bid1_size"] + out["ask1_size"]
    out["imbalance_l1"] = np.where(
        denom > 0,
        (out["bid1_size"] - out["ask1_size"]) / denom,
        0.0
    )

    out["microprice_l1"] = np.where(
        denom > 0,
        (out["ask1"] * out["bid1_size"] + out["bid1"] * out["ask1_size"]) / denom,
        out["mid"]
    )

    out["microprice_displacement"] = out["microprice_l1"] - out["mid"]
    out["microprice_displacement_ticks"] = out["microprice_displacement"] / config.tick_size
    out["microprice_displacement_spread_units"] = out["microprice_displacement"] / out["spread"].replace(0, np.nan)

    out["log_mid"] = np.log(out["mid"].clip(lower=1e-12))
    out["mid_return"] = out["log_mid"].diff().fillna(0.0)
    out["abs_mid_return"] = out["mid_return"].abs()

    out["quote_interval_seconds"] = out["timestamp"].diff().dt.total_seconds().fillna(out["quote_interval_seconds"].median())
    out["quote_interval_seconds"] = out["quote_interval_seconds"].clip(lower=1e-3)
    out["quote_intensity"] = 1.0 / out["quote_interval_seconds"]

    w = config.rolling_window
    out["rolling_mid_vol"] = out["mid_return"].rolling(w).std().fillna(method="bfill")
    out["rolling_abs_return"] = out["abs_mid_return"].rolling(w).mean().fillna(method="bfill")
    out["rolling_spread_ticks"] = out["spread_ticks"].rolling(w).mean().fillna(method="bfill")
    out["rolling_imbalance_l1"] = out["imbalance_l1"].rolling(w).mean().fillna(method="bfill")
    out["rolling_microprice_displacement"] = out["microprice_displacement_ticks"].rolling(w).mean().fillna(method="bfill")

    return out.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)


lob_l1 = add_l1_microprice_features(lob_raw, config)

lob_l1[[
    "timestamp",
    "bid1",
    "ask1",
    "bid1_size",
    "ask1_size",
    "mid",
    "imbalance_l1",
    "microprice_l1",
    "microprice_displacement_ticks"
]].head()

## 8. Multi-level LOB features

If levels 2 to $N$ are available, compute:

1. level-by-level imbalance;
2. unweighted depth imbalance;
3. weighted depth imbalance;
4. total bid/ask depth;
5. near-book versus far-book liquidity.

These features are **not available** in L1-only data.

In [ ]:
def add_multilevel_lob_features(df: pd.DataFrame, config: LOBConfig) -> pd.DataFrame:
    out = df.copy()
    L = config.n_levels

    bid_size_cols = [f"bid{i}_size" for i in range(1, L + 1)]
    ask_size_cols = [f"ask{i}_size" for i in range(1, L + 1)]

    has_multilevel = all(c in out.columns for c in bid_size_cols + ask_size_cols)

    out["has_multilevel_lob"] = has_multilevel

    if not has_multilevel:
        out["imbalance_lob"] = out["imbalance_l1"]
        out["weighted_imbalance_lob"] = out["imbalance_l1"]
        out["total_bid_depth"] = out["bid1_size"]
        out["total_ask_depth"] = out["ask1_size"]
        out["book_pressure"] = out["imbalance_l1"]
        return out

    bid_depth = out[bid_size_cols].sum(axis=1)
    ask_depth = out[ask_size_cols].sum(axis=1)

    out["total_bid_depth"] = bid_depth
    out["total_ask_depth"] = ask_depth
    out["imbalance_lob"] = (bid_depth - ask_depth) / (bid_depth + ask_depth).replace(0, np.nan)

    weights = np.array([1.0 / i for i in range(1, L + 1)])
    weighted_bid = out[bid_size_cols].to_numpy() @ weights
    weighted_ask = out[ask_size_cols].to_numpy() @ weights

    out["weighted_bid_depth"] = weighted_bid
    out["weighted_ask_depth"] = weighted_ask
    out["weighted_imbalance_lob"] = (weighted_bid - weighted_ask) / np.maximum(weighted_bid + weighted_ask, 1e-12)

    for level in range(1, L + 1):
        b = out[f"bid{level}_size"]
        a = out[f"ask{level}_size"]
        out[f"imbalance_level_{level}"] = (b - a) / (b + a).replace(0, np.nan)

    near_bid = out[[f"bid{i}_size" for i in range(1, min(3, L) + 1)]].sum(axis=1)
    near_ask = out[[f"ask{i}_size" for i in range(1, min(3, L) + 1)]].sum(axis=1)
    far_bid = out[[f"bid{i}_size" for i in range(min(3, L) + 1, L + 1)]].sum(axis=1) if L > 3 else 0
    far_ask = out[[f"ask{i}_size" for i in range(min(3, L) + 1, L + 1)]].sum(axis=1) if L > 3 else 0

    out["near_book_imbalance"] = (near_bid - near_ask) / np.maximum(near_bid + near_ask, 1e-12)

    if L > 3:
        out["far_book_imbalance"] = (far_bid - far_ask) / np.maximum(far_bid + far_ask, 1e-12)
    else:
        out["far_book_imbalance"] = 0.0

    out["book_pressure"] = 0.60 * out["imbalance_l1"] + 0.40 * out["weighted_imbalance_lob"]

    return out.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)


lob = add_multilevel_lob_features(lob_l1, config)

lob[[
    "imbalance_l1",
    "imbalance_lob",
    "weighted_imbalance_lob",
    "near_book_imbalance",
    "far_book_imbalance",
    "book_pressure"
]].head()

## 9. Short-horizon labels

For horizon $h$:

$$
\Delta M_{t,h}=M_{t+h}-M_t
$$

Direction label:

$$
direction_t=
\begin{cases}
1, & \Delta M_{t,h}>0\\
0, & \Delta M_{t,h}=0\\
-1, & \Delta M_{t,h}<0
\end{cases}
$$

We also create a binary move label for observations where the mid moves:

$$
y_t = \mathbb 1(\Delta M_{t,h}>0)
$$

In [ ]:
def add_prediction_labels(df: pd.DataFrame, config: LOBConfig) -> pd.DataFrame:
    out = df.copy()
    h = config.prediction_horizon_ticks

    out["future_mid"] = out["mid"].shift(-h)
    out["future_mid_change"] = out["future_mid"] - out["mid"]
    out["future_mid_change_ticks"] = out["future_mid_change"] / config.tick_size

    out["direction_3class"] = np.sign(out["future_mid_change_ticks"]).astype(float)
    out["is_move"] = out["direction_3class"] != 0
    out["direction_binary_up"] = (out["future_mid_change_ticks"] > 0).astype(float)

    out = out.dropna(subset=["future_mid"]).reset_index(drop=True)

    return out


lob = add_prediction_labels(lob, config)

label_summary = pd.Series({
    "n_rows": len(lob),
    "horizon_ticks": config.prediction_horizon_ticks,
    "move_fraction": lob["is_move"].mean(),
    "up_fraction_all": lob["direction_binary_up"].mean(),
    "up_fraction_conditional_on_move": lob.loc[lob["is_move"], "direction_binary_up"].mean()
})

label_summary

In [ ]:
plt.figure(figsize=(10, 5))
lob["direction_3class"].value_counts().sort_index().plot(kind="bar")
plt.title("Short-Horizon Mid-Price Direction Labels")
plt.xlabel("Direction: -1 down, 0 no move, 1 up")
plt.ylabel("Count")
plt.show()

## 10. Feature/target relationship diagnostics

Before fitting a classifier, inspect whether imbalance and microprice displacement have any relationship with future direction.

We bucket features and compute future up probability.

This is descriptive, not proof of alpha.

In [ ]:
def bucket_diagnostic(df, feature, target="direction_binary_up", move_only=True, q=10):
    work = df[df["is_move"]].copy() if move_only else df.copy()

    work["bucket"] = pd.qcut(work[feature], q=q, duplicates="drop")

    out = (
        work
        .groupby("bucket", observed=True)
        .agg(
            n=("direction_binary_up", "count"),
            mean_feature=(feature, "mean"),
            up_probability=(target, "mean"),
            mean_future_change_ticks=("future_mid_change_ticks", "mean")
        )
        .reset_index(drop=True)
    )

    return out


imbalance_bucket = bucket_diagnostic(lob, "imbalance_l1")
microprice_bucket = bucket_diagnostic(lob, "microprice_displacement_ticks")
lob_bucket = bucket_diagnostic(lob, "weighted_imbalance_lob")

imbalance_bucket

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(imbalance_bucket["mean_feature"], imbalance_bucket["up_probability"], marker="o", label="L1 imbalance")
plt.plot(microprice_bucket["mean_feature"], microprice_bucket["up_probability"], marker="x", label="Microprice displacement")
plt.axhline(0.5, linestyle="--")
plt.title("Feature Buckets vs Future Up Probability, Move Events Only")
plt.xlabel("Bucket mean feature value")
plt.ylabel("P(up | bucket)")
plt.legend()
plt.show()

## 11. Chronological split

We split chronologically.

No random shuffling is used.

Train set is used for fitting and scaling.

Test set is used only for final evaluation.

In [ ]:
split_idx = int(len(lob) * config.train_fraction)

train = lob.iloc[:split_idx].copy()
test = lob.iloc[split_idx:].copy()

pd.Series({
    "n_total": len(lob),
    "n_train": len(train),
    "n_test": len(test),
    "train_start": train["timestamp"].iloc[0],
    "train_end": train["timestamp"].iloc[-1],
    "test_start": test["timestamp"].iloc[0],
    "test_end": test["timestamp"].iloc[-1]
})

## 12. Model feature set

We compare:

### L1-only feature set

Compatible with top-of-book data:

- L1 imbalance;
- microprice displacement;
- spread;
- quote intensity;
- rolling mid volatility;
- rolling L1 imbalance.

### LOB feature set

Requires multi-level depth:

- L1 features;
- LOB imbalance;
- weighted LOB imbalance;
- near/far book imbalance;
- total bid/ask depth;
- book pressure.

In [ ]:
l1_feature_cols = [
    "imbalance_l1",
    "microprice_displacement_ticks",
    "microprice_displacement_spread_units",
    "spread_ticks",
    "relative_spread",
    "quote_intensity",
    "rolling_mid_vol",
    "rolling_abs_return",
    "rolling_spread_ticks",
    "rolling_imbalance_l1",
    "rolling_microprice_displacement"
]

lob_feature_cols = l1_feature_cols + [
    "imbalance_lob",
    "weighted_imbalance_lob",
    "near_book_imbalance",
    "far_book_imbalance",
    "total_bid_depth",
    "total_ask_depth",
    "weighted_bid_depth",
    "weighted_ask_depth",
    "book_pressure"
]

target_col = "direction_binary_up"

# Use move-only rows for binary up/down prediction.
train_move = train[train["is_move"]].copy()
test_move = test[test["is_move"]].copy()

len(train_move), len(test_move)

## 13. Train-only standardisation

We standardise features using train data only:

$$
z=\frac{x-\mu_{\text{train}}}{\sigma_{\text{train}}}
$$

This avoids leaking test distribution information.

In [ ]:
def fit_feature_scaler(df, cols):
    mean = df[cols].mean()
    std = df[cols].std(ddof=1).replace(0, 1.0)
    return mean, std


def transform_features(df, cols, mean, std):
    return ((df[cols] - mean) / std).replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy()


l1_mean, l1_std = fit_feature_scaler(train_move, l1_feature_cols)
lob_mean, lob_std = fit_feature_scaler(train_move, lob_feature_cols)

X_train_l1 = transform_features(train_move, l1_feature_cols, l1_mean, l1_std)
X_test_l1 = transform_features(test_move, l1_feature_cols, l1_mean, l1_std)

X_train_lob = transform_features(train_move, lob_feature_cols, lob_mean, lob_std)
X_test_lob = transform_features(test_move, lob_feature_cols, lob_mean, lob_std)

y_train = train_move[target_col].astype(int).to_numpy()
y_test = test_move[target_col].astype(int).to_numpy()

X_train_l1.shape, X_train_lob.shape, y_train.mean(), y_test.mean()

## 14. Baseline probabilistic classifiers

We use simple baselines:

1. constant train up probability;
2. sign of L1 imbalance;
3. sign of microprice displacement;
4. logistic regression on L1 features;
5. logistic regression on LOB features;
6. random forest on LOB features if available.

The goal is not to maximise leaderboard performance.

The goal is to measure whether LOB/microprice features add incremental information.

In [ ]:
def evaluate_probabilistic_classifier(y_true, prob_up, label):
    prob_up = np.asarray(prob_up, dtype=float)
    pred = (prob_up >= 0.5).astype(int)

    out = {
        "model": label,
        "n": int(len(y_true)),
        "accuracy": float(np.mean(pred == y_true)),
        "up_rate_predicted": float(pred.mean()),
        "up_rate_actual": float(np.mean(y_true)),
        "brier": float(np.mean((prob_up - y_true) ** 2)),
    }

    if len(np.unique(y_true)) == 2 and np.std(prob_up) > 0:
        if SKLEARN_AVAILABLE:
            out["auc"] = float(roc_auc_score(y_true, prob_up))
        else:
            out["auc"] = np.nan
    else:
        out["auc"] = np.nan

    return out


train_up_prob = y_train.mean()

baseline_prob_constant = np.full(len(y_test), train_up_prob)
baseline_prob_l1_sign = np.where(test_move["imbalance_l1"].to_numpy() > 0, 0.55, 0.45)
baseline_prob_microprice_sign = np.where(test_move["microprice_displacement_ticks"].to_numpy() > 0, 0.55, 0.45)

model_reports = [
    evaluate_probabilistic_classifier(y_test, baseline_prob_constant, "constant_train_up_probability"),
    evaluate_probabilistic_classifier(y_test, baseline_prob_l1_sign, "sign_l1_imbalance_rule"),
    evaluate_probabilistic_classifier(y_test, baseline_prob_microprice_sign, "sign_microprice_rule"),
]

fitted_models = {}
test_probabilities = {
    "constant_train_up_probability": baseline_prob_constant,
    "sign_l1_imbalance_rule": baseline_prob_l1_sign,
    "sign_microprice_rule": baseline_prob_microprice_sign,
}

if SKLEARN_AVAILABLE:
    logit_l1 = LogisticRegression(max_iter=500, C=1.0)
    logit_l1.fit(X_train_l1, y_train)
    prob_l1 = logit_l1.predict_proba(X_test_l1)[:, 1]

    fitted_models["logistic_l1"] = logit_l1
    test_probabilities["logistic_l1"] = prob_l1
    model_reports.append(evaluate_probabilistic_classifier(y_test, prob_l1, "logistic_l1"))

    logit_lob = LogisticRegression(max_iter=500, C=1.0)
    logit_lob.fit(X_train_lob, y_train)
    prob_lob = logit_lob.predict_proba(X_test_lob)[:, 1]

    fitted_models["logistic_lob"] = logit_lob
    test_probabilities["logistic_lob"] = prob_lob
    model_reports.append(evaluate_probabilistic_classifier(y_test, prob_lob, "logistic_lob"))

    rf_lob = RandomForestClassifier(
        n_estimators=150,
        max_depth=5,
        min_samples_leaf=100,
        random_state=config.seed,
        n_jobs=-1
    )
    rf_lob.fit(X_train_lob, y_train)
    prob_rf = rf_lob.predict_proba(X_test_lob)[:, 1]

    fitted_models["random_forest_lob"] = rf_lob
    test_probabilities["random_forest_lob"] = prob_rf
    model_reports.append(evaluate_probabilistic_classifier(y_test, prob_rf, "random_forest_lob"))

model_report = pd.DataFrame(model_reports).sort_values("brier")

model_report

## 15. Calibration by probability bucket

A probabilistic model should be calibrated.

If the model predicts 60% up probability, the realised up frequency should be close to 60% over similar forecasts.

We bucket predicted probabilities and compare predicted vs realised frequencies.

In [ ]:
def calibration_table(y_true, prob, n_bins=10):
    df = pd.DataFrame({
        "y": y_true,
        "prob": prob
    })

    df["bucket"] = pd.qcut(df["prob"], q=n_bins, duplicates="drop")

    out = (
        df
        .groupby("bucket", observed=True)
        .agg(
            n=("y", "count"),
            mean_predicted_prob=("prob", "mean"),
            realized_up_frequency=("y", "mean")
        )
        .reset_index(drop=True)
    )

    return out


best_model_name = model_report.iloc[0]["model"]
best_prob = test_probabilities[best_model_name]

calibration = calibration_table(y_test, best_prob, n_bins=10)

best_model_name, calibration

In [ ]:
plt.figure(figsize=(7, 6))
plt.plot(calibration["mean_predicted_prob"], calibration["realized_up_frequency"], marker="o")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.title(f"Probability Calibration: {best_model_name}")
plt.xlabel("Mean predicted up probability")
plt.ylabel("Realised up frequency")
plt.show()

## 16. Feature importance and coefficient inspection

For logistic regression, coefficients are interpretable.

Positive coefficient means the feature increases predicted up probability, holding other features constant.

For random forests, feature importances are impurity-based and should be interpreted cautiously.

In [ ]:
importance_frames = []

if SKLEARN_AVAILABLE and "logistic_l1" in fitted_models:
    importance_frames.append(pd.DataFrame({
        "model": "logistic_l1",
        "feature": l1_feature_cols,
        "importance": fitted_models["logistic_l1"].coef_[0]
    }))

if SKLEARN_AVAILABLE and "logistic_lob" in fitted_models:
    importance_frames.append(pd.DataFrame({
        "model": "logistic_lob",
        "feature": lob_feature_cols,
        "importance": fitted_models["logistic_lob"].coef_[0]
    }))

if SKLEARN_AVAILABLE and "random_forest_lob" in fitted_models:
    importance_frames.append(pd.DataFrame({
        "model": "random_forest_lob",
        "feature": lob_feature_cols,
        "importance": fitted_models["random_forest_lob"].feature_importances_
    }))

feature_importance = pd.concat(importance_frames, ignore_index=True) if importance_frames else pd.DataFrame()

feature_importance.head()

In [ ]:
if not feature_importance.empty:
    top_importance = (
        feature_importance
        .assign(abs_importance=lambda x: x["importance"].abs())
        .sort_values(["model", "abs_importance"], ascending=[True, False])
        .groupby("model")
        .head(10)
    )

    for model, group in top_importance.groupby("model"):
        plt.figure(figsize=(10, 6))
        group = group.sort_values("importance")
        plt.barh(group["feature"], group["importance"])
        plt.title(f"Top Feature Importance: {model}")
        plt.xlabel("Importance / coefficient")
        plt.ylabel("Feature")
        plt.show()
else:
    print("Feature importance unavailable because sklearn is not installed.")

## 17. Economic diagnostic: microprice edge by probability bucket

A classifier can have statistical accuracy but no economic edge.

We estimate average future tick move by predicted-probability bucket:

$$
E[\Delta M_{t,h} \mid \hat p_t \in bucket]
$$

If high predicted up probability does not correspond to positive future mid move, the model is not useful.

In [ ]:
economic_df = test_move[[
    "timestamp",
    "mid",
    "spread",
    "future_mid_change_ticks",
    "direction_binary_up"
]].copy()

for name, prob in test_probabilities.items():
    economic_df[f"prob_{name}"] = prob

economic_df["best_prob"] = best_prob
economic_df["prob_bucket"] = pd.qcut(economic_df["best_prob"], q=10, duplicates="drop")

economic_bucket = (
    economic_df
    .groupby("prob_bucket", observed=True)
    .agg(
        n=("future_mid_change_ticks", "count"),
        mean_prob=("best_prob", "mean"),
        realized_up_frequency=("direction_binary_up", "mean"),
        mean_future_change_ticks=("future_mid_change_ticks", "mean"),
        median_future_change_ticks=("future_mid_change_ticks", "median")
    )
    .reset_index(drop=True)
)

economic_bucket

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(economic_bucket["mean_prob"], economic_bucket["mean_future_change_ticks"], marker="o")
plt.axhline(0, linestyle="--")
plt.title("Predicted Probability vs Future Mid Change")
plt.xlabel("Mean predicted up probability")
plt.ylabel("Mean future mid change, ticks")
plt.show()

## 18. Toy execution signal with spread cost

A simple diagnostic trading rule:

$$
position_t =
\begin{cases}
+1, & \hat p_t > 0.55\\
-1, & \hat p_t < 0.45\\
0, & \text{otherwise}
\end{cases}
$$

Gross tick PnL:

$$
position_t \cdot \Delta M_{t,h}
$$

Cost proxy:

$$
cost = c \cdot spread_t \cdot |position_t-position_{t-1}|
$$

This is not a real execution simulation. It is a cost-sensitivity diagnostic.

In [ ]:
def toy_microprice_strategy(economic_df, prob_col, upper=0.55, lower=0.45, cost_fraction_of_spread=0.25):
    out = economic_df.copy()

    p = out[prob_col].to_numpy()

    position = np.where(p > upper, 1.0, np.where(p < lower, -1.0, 0.0))
    turnover = np.abs(np.diff(np.r_[0.0, position]))

    gross_ticks = position * out["future_mid_change_ticks"].to_numpy()
    cost_ticks = cost_fraction_of_spread * out["spread"].to_numpy() * turnover
    net_ticks = gross_ticks - cost_ticks

    out["position"] = position
    out["turnover"] = turnover
    out["gross_pnl_ticks"] = gross_ticks
    out["cost_ticks"] = cost_ticks
    out["net_pnl_ticks"] = net_ticks
    out["cum_net_pnl_ticks"] = np.cumsum(net_ticks)

    summary = {
        "prob_col": prob_col,
        "upper_threshold": upper,
        "lower_threshold": lower,
        "cost_fraction_of_spread": cost_fraction_of_spread,
        "trade_fraction": float(np.mean(position != 0)),
        "mean_position": float(np.mean(position)),
        "mean_turnover": float(np.mean(turnover)),
        "gross_pnl_ticks": float(np.sum(gross_ticks)),
        "net_pnl_ticks": float(np.sum(net_ticks)),
        "mean_net_pnl_per_event_ticks": float(np.mean(net_ticks)),
        "pnl_vol_ticks": float(np.std(net_ticks, ddof=1)),
        "hit_rate_when_trading": float(np.mean(np.sign(gross_ticks[position != 0]) > 0)) if np.any(position != 0) else np.nan
    }

    return out, summary


strategy_rows = []
strategy_frames = []

for name in test_probabilities:
    prob_col = f"prob_{name}"
    strat, summary = toy_microprice_strategy(economic_df, prob_col)
    strat["model"] = name
    strategy_frames.append(strat)
    strategy_rows.append(summary | {"model": name})

strategy_results = pd.concat(strategy_frames, ignore_index=True)
strategy_summary = pd.DataFrame(strategy_rows).sort_values("net_pnl_ticks", ascending=False)

strategy_summary

In [ ]:
best_strategy_model = strategy_summary.iloc[0]["model"]
best_strategy = strategy_results[strategy_results["model"] == best_strategy_model]

plt.figure(figsize=(12, 6))
for model, group in strategy_results.groupby("model"):
    plt.plot(group["cum_net_pnl_ticks"].to_numpy(), label=model, alpha=0.85)
plt.title("Toy Strategy Net PnL, Ticks")
plt.xlabel("Move event index")
plt.ylabel("Cumulative net ticks")
plt.legend()
plt.show()

best_strategy_model

## 19. Cost sensitivity

Microstructure signals are especially sensitive to costs.

We vary the spread-cost assumption and observe whether any apparent edge survives.

In [ ]:
cost_grid = [0.0, 0.05, 0.10, 0.25, 0.50, 1.00]
cost_rows = []

for cost in cost_grid:
    for name in test_probabilities:
        prob_col = f"prob_{name}"
        _, summary = toy_microprice_strategy(
            economic_df,
            prob_col,
            cost_fraction_of_spread=cost
        )
        cost_rows.append(summary | {"model": name})

cost_sensitivity = pd.DataFrame(cost_rows)

cost_sensitivity.head()

In [ ]:
plt.figure(figsize=(10, 6))
for model, group in cost_sensitivity.groupby("model"):
    plt.plot(group["cost_fraction_of_spread"], group["net_pnl_ticks"], marker="o", label=model)
plt.axhline(0, linestyle="--")
plt.title("Toy Strategy Cost Sensitivity")
plt.xlabel("Cost fraction of spread per turnover")
plt.ylabel("Net PnL, ticks")
plt.legend()
plt.show()

## 20. Stability through time

A microstructure edge can disappear quickly.

We compute rolling accuracy and rolling average future tick move by prediction bucket.

This checks whether performance is concentrated in one lucky period.

In [ ]:
def rolling_model_diagnostics(economic_df, prob_col, window=1000, step=250):
    rows = []

    for start in range(0, len(economic_df) - window + 1, step):
        sub = economic_df.iloc[start:start+window].copy()
        prob = sub[prob_col].to_numpy()
        pred = (prob >= 0.5).astype(int)
        y = sub["direction_binary_up"].astype(int).to_numpy()

        rows.append({
            "start": start,
            "end": start + window,
            "timestamp_start": sub["timestamp"].iloc[0],
            "timestamp_end": sub["timestamp"].iloc[-1],
            "accuracy": float(np.mean(pred == y)),
            "brier": float(np.mean((prob - y) ** 2)),
            "mean_future_change_long_prob": float(sub.loc[prob > 0.55, "future_mid_change_ticks"].mean()) if np.any(prob > 0.55) else np.nan,
            "trade_fraction": float(np.mean((prob > 0.55) | (prob < 0.45)))
        })

    return pd.DataFrame(rows)


rolling_diag = rolling_model_diagnostics(economic_df, f"prob_{best_model_name}", window=1000, step=250)

rolling_diag.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(rolling_diag["timestamp_end"], rolling_diag["accuracy"], marker="o", label="rolling accuracy")
plt.axhline(0.5, linestyle="--")
plt.title(f"Rolling Accuracy: {best_model_name}")
plt.xlabel("Window end")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(rolling_diag["timestamp_end"], rolling_diag["mean_future_change_long_prob"], marker="o")
plt.axhline(0, linestyle="--")
plt.title(f"Rolling Mean Future Change When Long Signal: {best_model_name}")
plt.xlabel("Window end")
plt.ylabel("Mean future change, ticks")
plt.show()

## 21. L1-only versus LOB feature comparison

If LOB features improve performance beyond L1 features, we should see improvement from:

```text
logistic_l1
```

to:

```text
logistic_lob
```

But that improvement must be judged out of sample and after costs.

If the improvement is tiny, the extra data complexity may not be worth it.

In [ ]:
l1_vs_lob = model_report[model_report["model"].isin(["logistic_l1", "logistic_lob", "random_forest_lob"])].copy()

l1_vs_lob

## 22. Practical checklist for LOB imbalance research

Before trusting imbalance or microprice signals, check:

1. **Data scope**  
   Is it L1 only or true multi-level LOB?

2. **Timestamp alignment**  
   Are features known before the prediction horizon begins?

3. **No future depth**  
   Are rolling features backward-looking only?

4. **Spread state**  
   Does the signal work in wide-spread and tight-spread regimes?

5. **No-move dominance**  
   Are no-move ticks handled honestly?

6. **Cost sensitivity**  
   Does the signal survive spread and turnover costs?

7. **Stability**  
   Is performance stable through time?

8. **Calibration**  
   Are predicted probabilities calibrated?

9. **Execution feasibility**  
   Can orders actually be filled at assumed prices?

10. **Book depth limitation**  
   If only L1 is available, do not claim full order-book imbalance.

## 23. Saving outputs

The notebook saves:

1. synthetic LOB snapshots;
2. engineered L1 and LOB feature table;
3. short-horizon labels;
4. bucket diagnostics;
5. model evaluation reports;
6. calibration table;
7. feature importance;
8. economic bucket diagnostics;
9. toy strategy results;
10. cost sensitivity;
11. rolling diagnostics;
12. manifest.

In [ ]:
output_dir = Path("data/processed/lob_imbalance_microprice_v1")
output_dir.mkdir(parents=True, exist_ok=True)

raw_path = output_dir / "synthetic_lob_snapshots.csv"
features_path = output_dir / "lob_microprice_features.csv"
label_summary_path = output_dir / "label_summary.csv"
imbalance_bucket_path = output_dir / "imbalance_bucket_diagnostic.csv"
microprice_bucket_path = output_dir / "microprice_bucket_diagnostic.csv"
model_report_path = output_dir / "model_report.csv"
calibration_path = output_dir / "calibration_table.csv"
feature_importance_path = output_dir / "feature_importance.csv"
economic_bucket_path = output_dir / "economic_bucket_diagnostic.csv"
strategy_summary_path = output_dir / "toy_strategy_summary.csv"
cost_sensitivity_path = output_dir / "cost_sensitivity.csv"
rolling_diag_path = output_dir / "rolling_diagnostics.csv"
manifest_path = output_dir / "manifest.json"

lob_raw.to_csv(raw_path, index=False)
lob.to_csv(features_path, index=False)
label_summary.to_frame("value").to_csv(label_summary_path)
imbalance_bucket.to_csv(imbalance_bucket_path, index=False)
microprice_bucket.to_csv(microprice_bucket_path, index=False)
model_report.to_csv(model_report_path, index=False)
calibration.to_csv(calibration_path, index=False)

if not feature_importance.empty:
    feature_importance.to_csv(feature_importance_path, index=False)
else:
    pd.DataFrame([{"note": "feature importance unavailable because sklearn is not installed"}]).to_csv(feature_importance_path, index=False)

economic_bucket.to_csv(economic_bucket_path, index=False)
strategy_summary.to_csv(strategy_summary_path, index=False)
cost_sensitivity.to_csv(cost_sensitivity_path, index=False)
rolling_diag.to_csv(rolling_diag_path, index=False)

manifest = {
    "dataset_name": "lob_imbalance_microprice_outputs",
    "schema_version": "lob_imbalance_microprice_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "sklearn_available": SKLEARN_AVAILABLE,
    "l1_feature_cols": l1_feature_cols,
    "lob_feature_cols": lob_feature_cols,
    "best_model_by_brier": model_report.sort_values("brier").iloc[0].to_dict(),
    "best_strategy_by_net_ticks": strategy_summary.sort_values("net_pnl_ticks", ascending=False).iloc[0].to_dict(),
    "label_summary": label_summary.to_dict(),
    "notes": [
        "Notebook demonstrates both L1-compatible features and multi-level LOB features.",
        "Top-of-book imbalance uses bid1_size and ask1_size only.",
        "Full LOB imbalance requires multi-level depth and should not be claimed from L1-only data.",
        "Short-horizon labels use future mid changes at fixed tick horizon.",
        "Binary models are trained only on move events to avoid no-move dominance.",
        "Toy execution signal is a diagnostic, not a realistic execution simulator.",
        "Cost sensitivity and rolling stability are included to avoid overclaiming microstructure alpha."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

raw_path, features_path, model_report_path, strategy_summary_path, manifest_path

## 24. Limitations

### 24.1 Synthetic LOB

The notebook uses a simplified synthetic LOB.

Real exchange data has queue priority, cancellations, hidden liquidity, market orders, matching rules, and session-specific behaviour.

### 24.2 L1 versus full LOB

Top-of-book imbalance is not the same as full LOB imbalance.

If only L1 is available, deeper book pressure is unknown.

### 24.3 No queue-position modelling

Microprice does not tell us whether our order would be filled.

Execution modelling requires queue assumptions.

### 24.4 No event-type data

The notebook does not distinguish limit order arrivals, cancellations, and trades.

Those event types can contain important information.

### 24.5 No latency model

A live signal must account for data feed latency, decision latency, and exchange latency.

### 24.6 No market impact

The toy strategy assumes no impact.

Real execution can move the book.

### 24.7 Synthetic predictive pressure

The synthetic data includes weak pressure by construction.

Real predictive relationships may be weaker, unstable, or absent after costs.

## 25. Research frontier and extensions

Important extensions include:

1. **Queue imbalance**  
   Model queue sizes and expected depletion.

2. **Order-flow imbalance**  
   Use trade signs and quote updates to estimate buying/selling pressure.

3. **Hawkes processes**  
   Model clustered arrivals of trades, quotes, and cancellations.

4. **DeepLOB-style models**  
   Use CNN/LSTM/attention on multi-level order-book tensors.

5. **Survival models for price moves**  
   Predict time to next mid-price move.

6. **Regime-conditioned microprice**  
   Combine GMM-HMM tick regimes with imbalance features.

7. **Execution-aware modelling**  
   Convert microprice features into order placement and participation decisions.

8. **Cross-contract futures microstructure**  
   Study lead-lag between main and deferred contracts.

9. **Chinese futures L1/L2 extension**  
   Include night sessions, price limits, product-specific tick sizes, and contract rolls.

10. **Realistic backtesting**  
   Add latency, slippage, queue priority, and exchange fee assumptions.

## 26. Suggested follow-up notebooks

This notebook naturally leads to:

1. **03_09_information_coefficient_analysis**  
   Evaluate imbalance signals as predictive factors.

2. **03_10_tree_models_for_return_prediction**  
   Use microstructure features in nonlinear ML models.

3. **05_03_transaction_costs_slippage_latency**  
   Make cost assumptions more realistic.

4. **06_04_hawkes_process_order_flow**  
   Model quote/trade event clustering.

5. **06_08_limit_order_book_replay_engine**  
   Build a more realistic LOB backtest.

6. **07_03_chinese_futures_l1_microstructure_alpha**  
   Capstone using L1 bid/ask, microprice, regimes, and execution diagnostics.

## 27. Summary

This notebook implemented LOB imbalance and microprice research.

It showed how to:

1. simulate synthetic L5 order-book snapshots;
2. compute L1 mid, spread, imbalance, and microprice;
3. compute multi-level LOB imbalance;
4. create short-horizon mid-price labels;
5. diagnose feature buckets against future direction;
6. compare L1-only and LOB feature sets;
7. fit simple probabilistic classifiers;
8. inspect calibration and feature importance;
9. evaluate economic buckets;
10. run a toy cost-sensitive execution signal;
11. test stability through time;
12. save outputs and diagnostics.

The key computational takeaway:

> Microprice and imbalance features are easy to compute but hard to validate honestly.

The key financial takeaway:

> LOB imbalance may contain short-horizon information, but any apparent edge must survive no-move handling, timestamp alignment, spread costs, and stability checks.

## 28. Further reading

- Cont, Stoikov, and Talreja on stochastic order-book dynamics.
- Gould et al., "Limit Order Books."
- Cartea, Jaimungal, and Penalva, *Algorithmic and High-Frequency Trading.*
- Bouchaud, Farmer, and Lillo on price impact and order books.
- Literature on microprice, queue imbalance, order-flow imbalance, and DeepLOB-style neural models.